# Soup Kernel Challenge: debug runner (generated from templates/kaggle_notebook.ipynb by tools/build_notebook.py)

`MODE`: `quick` (pytest + one solution run), `full` (base + solution x2 + checks), `full_profile` (full + profilers of solution and base).
Statuses: `/kaggle/working/out/status.txt`; logs and json: `/kaggle/working/out/`.


In [ ]:
REPO_URL = "https://github.com/alikhan-kai/decentrathon.git"
BRANCH = "main"
COMMIT = ""  # empty = tip of BRANCH
MODE = "quick"


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
names = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                       capture_output=True, text=True).stdout
assert "Tesla T4" in names, "Not a Tesla T4: " + names


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest


In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", "-b", BRANCH, REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
import glob, json, shutil, subprocess, sys
from pathlib import Path
REPO = Path("/kaggle/working/repo")
OUT = Path("/kaggle/working/out")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "status.txt").write_text("")
ENV = dict(os.environ)

def stage(name, cmd, env=None):
    """Run one stage, keep its log, record the status; never raises."""
    with open(OUT / f"{name}.log", "w") as log:
        code = subprocess.run(cmd, cwd=REPO, env=env or ENV, stdout=log,
                              stderr=subprocess.STDOUT).returncode
    line = f"{name}: {'ok' if code == 0 else 'FAIL (exit %d)' % code}"
    with open(OUT / "status.txt", "a") as f:
        f.write(line + "\n")
    print(line)
    if code:
        print("".join(open(OUT / f"{name}.log").readlines()[-60:]))
    return code


In [ ]:
stage("prepare_data", ["python", "task/prepare_data.py"])


In [ ]:
def pytests():
    stage("pytest_solution", ["python", "-m", "pytest", "solution", "-q"])
    stage("pytest_tests", ["python", "-m", "pytest", "tests", "-q"],
          env=dict(ENV, SOUP_APPLY_SOLUTION="1"))

def base_profile():
    """Profile of the BASE (run_task profiles only the solution). Same env as run_task."""
    import yaml
    src = REPO / "runs/base_1/task.yaml"
    if not src.exists():
        found = sorted(glob.glob(str(REPO / "runs/**/task.yaml"), recursive=True))
        print("runs/base_1/task.yaml missing; task.yaml files found:", found)
        (OUT / "status.txt").open("a").write("base_profile: FAIL (no runs/base_1/task.yaml)\n")
        return
    run_dir = REPO / "runs/base_profile"
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg = yaml.safe_load(src.read_text())
    cfg["output"] = str(run_dir / "adapter")  # do not overwrite the measured base adapter
    (run_dir / "task.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
    sys.path.insert(0, str(REPO / "harness"))
    from common import OFFLINE_ENV
    env = dict(OFFLINE_ENV, SOUP_DB_PATH=str(run_dir / "experiments.db"),
               TRITON_CACHE_DIR=str(run_dir / "triton_cache"),
               TORCHINDUCTOR_CACHE_DIR=str(run_dir / "inductor_cache"))
    stage("base_profile", ["python", "harness/_train.py", "--config", str(run_dir / "task.yaml"),
                           "--metrics", str(OUT / "base_profile_metrics.json"),
                           "--profile", str(OUT / "base_profile.json")], env=env)

if MODE == "quick":
    pytests()
    stage("run_task_solution", ["python", "harness/run_task.py", "--only", "solution",
                                "--out", "runs_try"])
    mp = REPO / "runs_try/solution_1/metrics.json"
    if mp.exists():
        m = json.load(open(mp))
        for k in ("triton_launches", "own_kernel_ms", "timed_loop_s", "torch_max_reserved_mib"):
            print(k, "=", m.get(k))
elif MODE in ("full", "full_profile"):
    if MODE == "full_profile":
        pytests()
    flags = ["--profile"] if MODE == "full_profile" else []
    stage("run_task_full", ["python", "harness/run_task.py", "--repeats", "2", *flags])
    stage("check_rules", ["python", "harness/check_rules.py"])
    if MODE == "full_profile":
        base_profile()
else:
    raise SystemExit("MODE must be quick, full or full_profile")


In [ ]:
for pattern in ("runs*/**/train.log", "runs*/eval_*.log", "runs*/result.json",
                "runs*/checks.json", "runs*/**/metrics.json", "runs*/solution_profile/profile.json",
                "runs/base_profile/train.log"):
    for p in glob.glob(str(REPO / pattern), recursive=True):
        rel = Path(p).relative_to(REPO)
        dst = OUT / "files" / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(p, dst)
shutil.rmtree(REPO, ignore_errors=True)  # keep the kernel output small
print((OUT / "status.txt").read_text())
